In [ ]:
#GROUP BY FUNCTION TO GET DATA FOR A SPECIFIC department

In [ ]:
#df.columns()
#grouped = df.groupby("col_name")
#print(grouped)
#print(grouped.groups)
#print(grouped.get_group("col_name"))

In [ ]:
#groupby()+count()
#result = df.groupby("col_name")["salary"].count()
#groupby()+apply()-->used if i want to use something customise
#roupby()+mean()


In [ ]:
#transform() --> gives a new value for each row .

In [ ]:
#nlargest()--> gives the n largest salaries

In [ ]:
#explode()--> breaks data into different rows

In [ ]:
#for merging we require some same fields
#we can have inner,outer,left,right join by default we use inner join if do not mention how

In [ ]:
#.shift()--> used for time series data.

In [ ]:
#status()--> true means when data changes but false means data changes

In [6]:
import pandas as pd
fixed_entries = [
{"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
"keywords": "fee cost price charge", "category": "billing"},
{"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
"keywords": "password reset login", "category": "account"},
{"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
"keywords": "hours timing open time", "category": "general"},
{"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
"keywords": "pay payment upi fee", "category": "billing"},
]
last_two = [4,7]
categories = ["billing","account","general"]
personalised_entries = [
    {
       "question":"how do i update my registered mobile number",
       "answer" : "by entering your new number in account settings",
       "keywords":"update mobile number",
       "category" : categories[4%3]
    },
    {
        "question": "how can i change my account email address",
        "answer": "Go to Account Settings and select the option to change your email address.",
        "keywords": "email change account",
        "category": categories[7 % 3]
    }
]
final_entries = fixed_entries + personalised_entries
df = pd.DataFrame(final_entries)
print(df)


                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4  how do i update my registered mobile number   
5    how can i change my account email address   

                                              answer                keywords  \
0                          The annual fee is Rs 500.   fee cost price charge   
1                   Go to Settings > Reset Password.    password reset login   
2                          We are open 9 AM to 5 PM.  hours timing open time   
3         You can pay via UPI, card, or net banking.     pay payment upi fee   
4    by entering your new number in account settings    update mobile number   
5  Go to Account Settings and select the option t...    email change account   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  account  
5  account

In [8]:
def score_hypothesis(query, df):
    query_words = set(query.lower().split())

    results = []

    for _, row in df.iterrows():

        # Combine question and keywords
        entry_text = row["question"] + " " + row["keywords"]
        entry_words = set(entry_text.lower().split())

        # Find common words
        matching_words = query_words.intersection(entry_words)

        # Calculate confidence
        confidence = len(matching_words) / len(query_words)

        # Add only matching entries
        if confidence > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "confidence": confidence
            })

    # Rank by confidence: highest first
    results = sorted(results, key=lambda x: x["confidence"], reverse=True)

    return results


# Example
query = "how can i pay the fee"

matches = score_hypothesis(query, df)

for match in matches:
    print(match)

{'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'confidence': 1.0}
{'question': 'how can i change my account email address', 'answer': 'Go to Account Settings and select the option to change your email address.', 'confidence': 0.5}
{'question': 'what is the annual fee', 'answer': 'The annual fee is Rs 500.', 'confidence': 0.3333333333333333}
{'question': 'how do i update my registered mobile number', 'answer': 'by entering your new number in account settings', 'confidence': 0.3333333333333333}
{'question': 'how to reset password', 'answer': 'Go to Settings > Reset Password.', 'confidence': 0.16666666666666666}


In [9]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]


result = same_category("account", df)

print(result)

1                          how to reset password
4    how do i update my registered mobile number
5      how can i change my account email address
Name: question, dtype: object


In [10]:
new_keyword = input("Enter new keyword: ")
df.loc[0, "keywords"] = df.loc[0, "keywords"] + " " + new_keyword
df.to_csv("47_faq_data.csv", index=False)

print(df)

Enter new keyword: amount
                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4  how do i update my registered mobile number   
5    how can i change my account email address   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4    by entering your new number in account settings   
5  Go to Account Settings and select the option t...   

                       keywords category  
0  fee cost price charge amount  billing  
1          password reset login  account  
2        hours timing open time  general  
3           pay payment upi fee  billing  
4          upda

In [11]:
category_count = df.groupby("category").size()

print(category_count)

category
account    3
billing    2
general    1
dtype: int64


In [14]:
def score_hypothesis(query, df):

    query_words = set(query.lower().split())

    results = []

    for _, row in df.iterrows():

        entry_text = row["question"] + " " + row["keywords"]
        entry_words = set(entry_text.lower().split())

        matching_words = query_words.intersection(entry_words)

        confidence = len(matching_words) / len(query_words)

        if confidence > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "confidence": confidence
            })

    # Sort all matches from highest to lowest confidence
    results = sorted(
        results,
        key=lambda x: x["confidence"],
        reverse=True
    )

    if len(results) == 0:
        return []

    # Find the highest score
    highest_score = results[0]["confidence"]

    # Return ALL entries with the highest score
    best_matches = []

    for result in results:
        if result["confidence"] == highest_score:
            best_matches.append(result)

    return best_matches
query = "fee"#query with a tie
matches = score_hypothesis("fee", df)

print("Results for query: fee")

for match in matches:
    print(match)

Results for query: fee
{'question': 'what is the annual fee', 'answer': 'The annual fee is Rs 500.', 'confidence': 1.0}
{'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'confidence': 1.0}


In [15]:
def score_hypothesis(query, df):

    query_words = set(query.lower().split())

    results = []

    for _, row in df.iterrows():

        entry_text = row["question"] + " " + row["keywords"]
        entry_words = set(entry_text.lower().split())

        matching_words = query_words.intersection(entry_words)

        confidence = len(matching_words) / len(query_words)

        if confidence > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "confidence": confidence
            })

    # Sort all matches from highest to lowest confidence
    results = sorted(
        results,
        key=lambda x: x["confidence"],
        reverse=True
    )

    if len(results) == 0:
        return []

    # Find the highest score
    highest_score = results[0]["confidence"]

    # Return ALL entries with the highest score
    best_matches = []

    for result in results:
        if result["confidence"] == highest_score:
            best_matches.append(result)

    return best_matches
  #query without a tie
matches = score_hypothesis("password reset", df)

print("Results for query: password reset")

for match in matches:
    print(match)

Results for query: password reset
{'question': 'how to reset password', 'answer': 'Go to Settings > Reset Password.', 'confidence': 1.0}
